# MS-prime

https://tskit.dev/msprime/docs/stable/quickstart.html

In [ ]:
import msprime
from IPython.display import SVG, display

In [ ]:
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

## First Tree

In [ ]:
n = 5
N = 1e3

ts = msprime.simulate(n,Ne=N/2) #N/2 due to diploidy
tree = ts.first()
tree.draw_svg(y_axis=True,size=(500,500))

## Time to the common ancestor

For each pair, display the time until their coalescence

In [ ]:
for i in range(n-1):
    for j in range(i+1,n):
        print(i,j,tree.tmrca(i,j))

Get all coalescence times:

In [ ]:
times = np.array(sorted([
    tree.time(u)
    for u in tree.nodes()
    if tree.num_children(u) > 1
]))

times

## Waiting times

In [ ]:
times = times - np.concatenate(([0], times[:-1]))
times

'Waiting times' are exponentially / geometric distributed.

In coalescence theory, we have $\binom{k}{2}$ possible pairs of lineages that may coalesce, each with probability $1/N$.
Hence, the waiting time $T_k$ is exponentially distributed with $$E[T_k] = \frac{N}{\binom{k}{2}} = \frac{2N}{k(k-1)}$$
And the total time to the common ancestor resolves to
$$E[T_{MRCA}] = \sum_k E[T_k] = \sum_k \frac{2N}{k(k-1)} = 2N(1-\frac{1}{n})$$
Let's simulate $5,000$ coalescence trees with sample size $n=5$ and plot the distribution of the waiting times $T_k$ and the time to the Most recent common ancestor $T_{MRCA}$.

In [ ]:
it = 5000
n = 5
N = 1e3

coalescence_times = []
TMRCAs =[]
for _ in tqdm(range(it)):
    ts = msprime.simulate(n,Ne=N/2)
    tree = ts.first()
    times = np.array(sorted([
        tree.time(u)
        for u in tree.nodes()
        if tree.num_children(u) > 1
    ]))
    times = times - np.concatenate(([0], times[:-1]))
    coalescence_times.append(times)
    TMRCAs.append(tree.time(tree.root))


In [ ]:
def plot_column_histograms(data, bins=100):
    data = np.array(data)
    n_cols = data.shape[1]

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    # handle case with only one column
    if n_cols == 1:
        axes = [axes]

    for i in range(n_cols):
        k = n_cols - i + 1
        lam = (k * (k - 1)) / (2 * N)

        
        axes[i].hist(data[:, i], bins=bins, color="skyblue", edgecolor="black",density=True)
        #axes[i].set_ylim(top=axes[i].get_ylim()[1])
        axes[i].set_ylim(top=0.01)
        #axes[i].set_title(f"Column {i}")
        axes[i].set_title(f"k={k}, E[T_k] = {np.round(1/lam,2)}, mean(T_k) = {np.round(np.mean(data[:,i]),2)}")
        axes[i].set_xlabel("Value")
        axes[i].set_ylabel("Frequency")
        
        #print(k,np.round(1/lam,2),np.round(np.mean(data[:,i]),2))

        # x-Werte für glatte Kurve
        x = np.linspace(0, max(data[:, i]), 200)
        
        # Exponentialverteilung: f(x) = λ * exp(-λx)
        y = lam * np.exp(-lam * x)
        
        # Plot der Kurve
        axes[i].plot(x, y, 'r-', lw=2, label=f"Exponential (λ={lam:.2f})")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_column_histograms(coalescence_times)

In [ ]:
mean_val = np.mean(TMRCAs)
exp_val = 2*N*(1-1/n)
plt.hist(TMRCAs,bins=200,density=True,alpha=0.8)
plt.axvline(mean_val, color="red", alpha=0.7, linestyle="--", linewidth=4, label=f"Mean = {mean_val:.2f}")
plt.axvline(exp_val, color="green", alpha=0.7,linestyle="--", linewidth=4, label=f"Expected = {exp_val:.2f}")
plt.title('TMRCA')
plt.legend()

## Mutation patterns

In [ ]:
n = 5
N = 1e3
mu = 2e-4

ts = msprime.simulate(n,Ne=N/2,mutation_rate=mu) #N/2 due to diploidy
tree = ts.first()
tree.draw_svg(y_axis=True,size=(500,500))

In [ ]:
snp_mat = ts.genotype_matrix()
snp_mat

#### Define three functions to determine $\theta_\pi$, $\theta_W$ and Tajima's D

In [ ]:
def theta_w(G):
    n = G.shape[1]  # number of samples
    S = G.shape[0]  # number of segregating sites

    a1 = np.sum(1 / np.arange(1, n))
    theta_w = S / a1 if a1 > 0 else 0

    return np.round(theta_w,3)


def theta_pi(G):
    n = G.shape[1]  # number of samples
    S = G.shape[0]  # number of segregating sites

    # allele counts per site
    p = np.sum(G, axis=1)  # derived allele count

    # theta_pi: average pairwise differences
    pi_per_site = 2 * p * (n - p) / (n * (n - 1))
    theta_pi = np.sum(pi_per_site)

    return np.round(theta_pi,3)


def tajimas_d(G):
    n = G.shape[1]
    S = G.shape[0]

    #theta_pi = theta_pi(G)
    #theta_w = theta_w(G)

    # constants
    a1 = np.sum(1 / np.arange(1, n))
    a2 = np.sum(1 / (np.arange(1, n) ** 2))

    b1 = (n + 1) / (3 * (n - 1))
    b2 = 2 * (n**2 + n + 3) / (9 * n * (n - 1))

    c1 = b1 - 1 / a1
    c2 = b2 - (n + 2) / (a1 * n) + a2 / (a1**2)

    e1 = c1 / a1
    e2 = c2 / (a1**2 + a2)

    # variance
    var_d = e1 * S + e2 * S * (S - 1)

    if var_d == 0:
        return np.nan

    D = (theta_pi(G) - theta_w(G)) / np.sqrt(var_d)
    return np.round(D,3)

In [ ]:
print(theta_w(snp_mat),theta_pi(snp_mat),tajimas_d(snp_mat))

Let's simulate again multiple scenarios, calculate $\theta_\pi$, $\theta_W$ and $D$

In [ ]:
it = 5000
n = 10
N = 1e3
mu = 1e-3

t_pi = []
t_W =[]
D = []
for _ in tqdm(range(it)):
    ts = msprime.simulate(n,Ne=N/2,mutation_rate=mu)
    snp_mat = ts.genotype_matrix()
    t_pi.append(theta_pi(snp_mat))
    t_W.append(theta_w(snp_mat))
    D.append(tajimas_d(snp_mat))

In [ ]:
plt.hist(D,bins=100,density=True)
plt.show()
plt.hist(t_pi, bins=20, density=True, alpha=0.5, color="red",label="t_pi")
plt.hist(t_W, bins=20, density=True, alpha=0.5, color="blue",label="t_W")
plt.legend()
plt.show()

In [ ]:
print(np.mean(t_pi),np.mean(t_W),mu*N*2)